# Attention Head Priority — JSD Similarity Analysis

Computes pairwise Jensen-Shannon Divergence (JSD) between all attention heads across a dataset, then outputs:
- A **similarity matrix** (`.npy` file)
- A **CSV** of heads-to-**keep** per layer at each similarity threshold

### Supported models
| Key | Model |
|-----|-------|
| `deitsmall` | `facebook/deit-small-patch16-224` |
| `dinobase` | `facebook/dino-vitb16` |
| `vitlarge` | `google/vit-large-patch16-224` |

### Supported datasets
| Key | Dataset |
|-----|--------|
| `mnist` | MNIST (test split) |
| `cifar100` | CIFAR-100 (test split) |
| `imagenet1k` | ImageNet-1k (validation split — requires HF login) |

> **ImageNet note:** you must run `huggingface-cli login` (or set `HF_TOKEN`) and have accepted the dataset terms at [https://huggingface.co/datasets/ILSVRC/imagenet-1k](https://huggingface.co/datasets/ILSVRC/imagenet-1k).

## 1 — Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate torchinfo torchvision matplotlib tqdm seaborn

## 2 — Core pipeline

In [ ]:
import gc
import csv
import torch
import numpy as np
import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from transformers import (
    AutoFeatureExtractor,
    DeiTForImageClassification,
    ViTForImageClassification,
    AutoModelForImageClassification,
)


# ─────────────────────────────────────────────────────────────────────────────
# Registry
# ─────────────────────────────────────────────────────────────────────────────

MODEL_REGISTRY = {
    "deitsmall": {
        "hf_id": "facebook/deit-small-patch16-224",
        "num_layers": 12,
        "num_heads": 6,
        "loader": "deit",
    },
    "dinobase": {
        "hf_id": "facebook/dino-vitb16",
        "num_layers": 12,
        "num_heads": 12,
        "loader": "vit",
    },
    "vitlarge": {
        "hf_id": "google/vit-large-patch16-224",
        "num_layers": 24,
        "num_heads": 16,
        "loader": "vit",
    },
}

DATASET_REGISTRY = {
    "mnist",
    "cifar100",
    "imagenet1k",
}


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _make_transform(image_size: int = 224) -> transforms.Compose:
    """Standard ViT preprocessing: resize → RGB → tensor → normalise."""
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.Grayscale(num_output_channels=3),   # no-op for RGB images
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])


def _load_dataset(dataset_name: str, num_samples: int, transform):
    """Return a DataLoader-ready Subset for the requested dataset."""
    dataset_name = dataset_name.lower()

    if dataset_name == "mnist":
        ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    elif dataset_name == "cifar100":
        ds = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

    elif dataset_name == "imagenet1k":
        from datasets import load_dataset as hf_load
        from PIL import Image
        import io

        hf_ds = hf_load("ILSVRC/imagenet-1k", split="validation", streaming=True, trust_remote_code=True)

        class _ImageNetWrapper(torch.utils.data.IterableDataset):
            def __init__(self, hf_iter, n, tfm):
                self.hf_iter = hf_iter
                self.n = n
                self.tfm = tfm

            def __iter__(self):
                for i, item in enumerate(self.hf_iter):
                    if i >= self.n:
                        break
                    img = item["image"].convert("RGB")
                    yield self.tfm(img), item["label"]

        return DataLoader(
            _ImageNetWrapper(hf_ds, num_samples, transform),
            batch_size=64,
            num_workers=0,
        ), num_samples

    else:
        raise ValueError(f"Unknown dataset '{dataset_name}'. Choose from: {DATASET_REGISTRY}")

    n = min(num_samples, len(ds))
    subset = Subset(ds, range(n))
    return DataLoader(subset, batch_size=64, shuffle=False, num_workers=2), n


def _load_model(model_key: str, device: torch.device):
    """Instantiate the requested model with attention/hidden-state outputs enabled."""
    cfg = MODEL_REGISTRY[model_key]
    hf_id = cfg["hf_id"]

    kwargs = dict(
        attn_implementation="eager",
        output_attentions=True,
        output_hidden_states=True,
        ignore_mismatched_sizes=True,
    )

    if cfg["loader"] == "deit":
        model = DeiTForImageClassification.from_pretrained(hf_id, **kwargs)
    else:
        model = AutoModelForImageClassification.from_pretrained(hf_id, **kwargs)

    model.to(device)
    model.eval()
    return model, cfg["num_layers"], cfg["num_heads"]


# ─────────────────────────────────────────────────────────────────────────────
# JSD accumulation
# ─────────────────────────────────────────────────────────────────────────────

def _accumulate_jsd(model, loader, num_layers, num_heads, device):
    """Stream batches through the model and accumulate the pairwise JSD matrix."""
    total_heads = num_layers * num_heads
    total_jsd = torch.zeros((total_heads, total_heads), dtype=torch.float64, device="cpu")
    n_images = 0

    with torch.no_grad():
        for images, _ in tqdm.tqdm(loader, desc="  Batches"):
            images = images.to(device)
            bs = images.shape[0]
            outputs = model(images)

            # attentions: tuple of (bs, num_heads, seq, seq), one per layer
            stacked = torch.stack(outputs.attentions)  # [L, bs, H, S, S]
            flat = stacked.permute(1, 0, 2, 3, 4).reshape(bs, total_heads, -1)  # [bs, TH, S*S]

            for i in range(bs):
                heads = flat[i]          # [TH, S*S]
                P = heads.unsqueeze(1)   # [TH, 1,  S*S]
                Q = heads.unsqueeze(0)   # [1,  TH, S*S]
                M = torch.clamp(0.5 * (P + Q), min=1e-12)
                P = torch.clamp(P, min=1e-12)
                Q = torch.clamp(Q, min=1e-12)

                kl_pm = torch.sum(P * (torch.log2(P) - torch.log2(M)), dim=-1)
                kl_qm = torch.sum(Q * (torch.log2(Q) - torch.log2(M)), dim=-1)
                total_jsd += (0.5 * kl_pm + 0.5 * kl_qm).cpu().double()
                n_images += 1

            del outputs, stacked, flat
            torch.cuda.empty_cache()
            gc.collect()

    avg_jsd = total_jsd / n_images
    similarity = (1.0 - avg_jsd).numpy()
    return similarity, n_images


# ─────────────────────────────────────────────────────────────────────────────
# Threshold → heads-to-KEEP (complement of removed)
# ─────────────────────────────────────────────────────────────────────────────

def _compute_keep_dicts(similarity, num_layers, num_heads, thresholds):
    """
    For each threshold: within each layer, mark the *higher-index* head in any
    pair that exceeds the threshold as redundant (to remove).  Return the
    complement — i.e. which heads to KEEP — as a dict keyed by layer index.
    """
    results = []   # list of (threshold, {layer: [keep_indices]})
    all_heads = set(range(num_heads))

    for thresh in thresholds:
        head_dict = {}
        for l in range(num_layers):
            start = l * num_heads
            to_remove = set()
            for i in range(start, start + num_heads):
                for j in range(i + 1, start + num_heads):
                    if similarity[i][j] > thresh:
                        to_remove.add(j - start)
            head_dict[l] = sorted(all_heads - to_remove)   # KEEP = complement
        results.append((thresh, head_dict))

    return results


# ─────────────────────────────────────────────────────────────────────────────
# Output writers
# ─────────────────────────────────────────────────────────────────────────────

def _save_similarity_matrix(similarity, out_path: Path):
    np.save(out_path, similarity)
    print(f"  ✅ Similarity matrix saved → {out_path}")


def _save_csv(threshold_results, num_layers, out_path: Path):
    """
    CSV layout:
      threshold | layer_0_keep | layer_1_keep | ... | layer_N_keep
    Each cell lists the kept head indices as a space-separated string.
    """
    layer_cols = [f"layer_{l}_keep" for l in range(num_layers)]
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["threshold"] + layer_cols)
        for thresh, head_dict in threshold_results:
            row = [f"{thresh:.2f}"] + [
                " ".join(map(str, head_dict[l])) for l in range(num_layers)
            ]
            writer.writerow(row)
    print(f"  ✅ CSV saved → {out_path}")


def _save_heatmap(similarity, n_images, num_heads, model_key, dataset_name, out_path: Path):
    total_heads = similarity.shape[0]
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        similarity, ax=ax, cmap="viridis", square=True, vmin=0, vmax=1,
        cbar_kws={"label": "Jensen–Shannon Similarity (1 − JSD)"},
    )
    ax.set_title(
        f"Pairwise Attention Head JSD Similarity\n"
        f"Model: {model_key}  |  Dataset: {dataset_name}  |  N={n_images}",
        fontsize=13, pad=12,
    )
    for tick in range(0, total_heads, num_heads):
        ax.axhline(tick, color="white", lw=0.7, ls="--")
        ax.axvline(tick, color="white", lw=0.7, ls="--")
    ax.set_xlabel(f"Global Head Index ({total_heads} total)", labelpad=8)
    ax.set_ylabel(f"Global Head Index ({total_heads} total)", labelpad=8)
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"  ✅ Heatmap saved → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────────────────────────────────────

def run_head_priority(
    model_name: str,
    dataset_name: str,
    num_samples: int = 5000,
    thresholds: list = None,
    output_dir: str = "./outputs",
):
    """
    Run the full attention-head JSD similarity pipeline.

    Parameters
    ----------
    model_name   : One of 'deitsmall', 'dinobase', 'vitlarge'
    dataset_name : One of 'mnist', 'cifar100', 'imagenet1k'
    num_samples  : How many images to evaluate (default 5000)
    thresholds   : List of float thresholds (default: 0.05, 0.10, …, 0.95)
    output_dir   : Where to write all output files
    """
    # ── Validate inputs ───────────────────────────────────────────────────────
    model_key = model_name.lower().strip()
    ds_key = dataset_name.lower().strip()

    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{model_name}'. Choose from: {list(MODEL_REGISTRY)}")
    if ds_key not in DATASET_REGISTRY:
        raise ValueError(f"Unknown dataset '{dataset_name}'. Choose from: {DATASET_REGISTRY}")

    if thresholds is None:
        thresholds = [round(t / 100, 2) for t in range(5, 100, 5)]

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = f"{model_key}_{ds_key}"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"  Model   : {model_key}  ({MODEL_REGISTRY[model_key]['hf_id']})")
    print(f"  Dataset : {ds_key}")
    print(f"  Samples : {num_samples}")
    print(f"  Device  : {device}")
    print(f"{'='*60}\n")

    # ── Load model ────────────────────────────────────────────────────────────
    print("[1/4] Loading model...")
    model, num_layers, num_heads = _load_model(model_key, device)
    total_heads = num_layers * num_heads
    print(f"      {num_layers} layers × {num_heads} heads = {total_heads} total heads")

    # ── Load dataset ──────────────────────────────────────────────────────────
    print("[2/4] Loading dataset...")
    transform = _make_transform(image_size=224)
    loader, actual_n = _load_dataset(ds_key, num_samples, transform)
    print(f"      Using {actual_n} samples")

    # ── Accumulate JSD ────────────────────────────────────────────────────────
    print("[3/4] Accumulating pairwise JSD...")
    similarity, n_images = _accumulate_jsd(model, loader, num_layers, num_heads, device)

    # ── Save similarity matrix ────────────────────────────────────────────────
    print("[4/4] Saving outputs...")
    _save_similarity_matrix(similarity, out_dir / f"{stem}_similarity.npy")
    _save_heatmap(similarity, n_images, num_heads, model_key, ds_key,
                  out_dir / f"{stem}_heatmap.png")

    # ── Compute and save CSV ──────────────────────────────────────────────────
    threshold_results = _compute_keep_dicts(similarity, num_layers, num_heads, thresholds)
    _save_csv(threshold_results, num_layers, out_dir / f"{stem}.csv")

    print(f"\n🎉 Done! All outputs written to: {out_dir.resolve()}")
    return similarity, threshold_results


## 3 — Run it

Change `model_name` and `dataset_name` as needed, then run the cell.

**Models:** `deitsmall` · `dinobase` · `vitlarge`  
**Datasets:** `mnist` · `cifar100` · `imagenet1k`

In [ ]:
similarity_matrix, threshold_results = run_head_priority(
    model_name="deitsmall",    # ← change me
    dataset_name="cifar100",   # ← change me
    num_samples=5000,          # how many images to evaluate
    # thresholds=[0.5, 0.7, 0.9],  # optional: override default 0.05–0.95 sweep
    output_dir="./outputs",
)


  Model   : deitsmall  (facebook/deit-small-patch16-224)
  Dataset : cifar100
  Samples : 5000
  Device  : cuda

[1/4] Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] You are using a model of type `vit` to instantiate a model of type `deit`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] DeiTForImageClassification LOAD REPORT from: facebook/deit-small-patch16-224
Key                                                         | Status     | 
------------------------------------------------------------+------------+-
vit.encoder.layer.{0...11}.layernorm_before.bias            | UNEXPECTED | 
vit.encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
vit.encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
vit.embeddings.patch_embeddings.projection.bias             | UNEXPECTED | 
vit.encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
vit.encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
vit.encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
vit.encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
vit.encoder.layer.{0...11}.attention.output.dense.bias      | UNEXPECTED | 
vit.encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED

      12 layers × 6 heads = 72 total heads
[2/4] Loading dataset...


 88%|████████▊ | 150M/169M [42:27<08:39, 37.5kB/s]

## 4 — (Optional) Preview the CSV

In [ ]:
import pandas as pd
df = pd.read_csv("./outputs/deitsmall_cifar100.csv")  # adjust filename if needed
df.head(10)

## 5 — (Optional) Download all outputs to your machine

In [ ]:
import shutil, os
shutil.make_archive("outputs", "zip", "./outputs")
from google.colab import files
files.download("outputs.zip")